# ConceptSFT × HarnessLLM: fine-tune Qwen3-4B sinh test harness bắt bug + học concept (Kaggle)

**Mục tiêu.** Fine-tune model 4B để sinh *test harness* (`generate_input_*` + `check_output`) **bắt bug** trong một
chương trình Python, rồi so: có học concept và không học concept, trên **cùng data, cùng budget, cùng judge**.

**Baseline paper:** HarnessLLM (Liu et al., NeurIPS'25, arXiv 2511.01104). Phần dùng lại *nguyên bản* từ paper:
- data SFT `Shiyu-Lab/HarnessLLM_SFT_Data` và eval `Shiyu-Lab/Testcase_*`
- prompt `prompts/harness.py` và judge `custom_verl/testcase_judge.compute_score` (clone repo, pin commit, **không sửa**)
- checkpoint cuối của họ `Shiyu-Lab/HarnessLLM_RL_Qwen3_4B` (SFT+RL), dùng làm mốc tham chiếu

**Concept learning, cơ chế mượn từ ConceptCoder** (`ConceptCoder_org/.../statement_level_sequential_multi_tasking.py`):
lấy hidden state cuối, **average-pool theo từng dòng** của chương trình, đưa qua `cbl = Linear(hidden, K)`, tính BCE với
nhãn concept per-line, `loss = LM + 0.2·BCE` (0.2 = `concept_weight` trong `run_conceptcoder_fine_tuning.sh`).
ConceptCoder gốc là classifier (VD/BP), full-FT ZeRO-3 trên C/C++, nên **không dùng lại trực tiếp** để sinh test.
Ở đây cơ chế đó được ghép vào một causal-LM sinh harness. Concept là 8 nhãn bug-prone cho Python, gán bằng AST
(tương tự 8 nhãn tree-sitter của ConceptCoder VD).

| VARIANT | Train | Prompt lúc infer | Ý nghĩa |
|---|---|---|---|
| `base` | – | prompt HarnessLLM | Qwen3-4B gốc |
| `harness_rl` | – | prompt HarnessLLM | checkpoint SFT+RL của paper (mốc trên) |
| `sft` | QLoRA trên data SFT của paper | prompt HarnessLLM | **baseline chính** = stage-1 của paper chạy với budget Kaggle |
| `concept_aux` | như `sft` + aux concept head | **giống `sft`** (không cần annotator lúc infer) | **phương pháp chính** |
| `placebo_aux` | như `concept_aux` nhưng nhãn concept bị xáo | giống `sft` | kiểm soát: lợi ích đến từ concept hay chỉ từ thêm loss |
| `concept_prompt` | như `sft`, prompt có thêm concept hints | prompt + hints | concept phía input (tuỳ chọn) |

**Metric** (đúng `scripts/eval.py` của họ): `PassGoodFailBad` = harness hợp lệ, pass bản đúng và **fail bản lỗi** (bắt được bug).
Kèm theo: `FailGood` (test sai, fail cả bản đúng), `GoodInput`, `FormatError`.

### Cách chạy trên Kaggle
1. **Settings ▸ Accelerator = GPU T4 ×2 (hoặc P100), Internet = ON**.
2. Mỗi phiên đặt `VARIANT` (và `SEED`) ở Cell 2. Gợi ý chia phiên (mỗi phiên ≤ ~5h):
   - Phiên 1: `base`, rồi `harness_rl` (chỉ eval, ~1h mỗi cái)
   - Phiên 2: `sft` · Phiên 3: `concept_aux` · Phiên 4: `placebo_aux` · (tuỳ chọn) `concept_prompt`
3. Cuối mỗi phiên **Save Version** để giữ `/kaggle/working/results_*.json` và `adapter_*`. Phiên sau **Add Data** = output
   các phiên trước. Cell 10 gộp mọi `results_*.json` lại và so sánh có kiểm định paired (McNemar + bootstrap).

> **Khác paper (phải nói rõ khi báo cáo):** paper full-FT 15 epoch, cutoff 16k, bf16, có thinking. Ở đây dùng QLoRA r=16,
> 1 epoch, ≤3k token, fp16, `THINK=0`, eval trên subset `N_EVAL` bài, decode tối đa 2048 token. Số tuyệt đối **không so
> được** với bảng của paper. So sánh hợp lệ là giữa các VARIANT với nhau (cùng data, budget, decode, judge), kể cả
> `harness_rl` khi decode dưới cùng budget.

In [ ]:
# === Cell 1: Cài đặt + clone HarnessLLM (pin commit, KHÔNG sửa code của họ) ===
import subprocess, sys, os
def sh(c):
    print("$", c); r = subprocess.run(c, shell=True, capture_output=True, text=True)
    if r.returncode: print(r.stdout[-1500:], r.stderr[-1500:])
    return r.returncode
sh("pip install -q -U 'transformers>=4.51' 'peft>=0.13' 'accelerate>=1.0' bitsandbytes datasets scipy")
# judge của HarnessLLM import `prctl` (python-prctl cần header libcap)
sh("(apt-get -qq update && apt-get -qq install -y libcap-dev) >/dev/null 2>&1; pip install -q python-prctl")

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
HARNESS_COMMIT = "83a58d4b50dec3f5a910b4b9762ae009dcd818f4"
HARNESS_DIR = os.path.join(WORK, "HarnessLLM")
if not os.path.isdir(HARNESS_DIR):
    sh(f"git clone -q https://github.com/UCSB-NLP-Chang/HarnessLLM.git {HARNESS_DIR}")
sh(f"cd {HARNESS_DIR} && git checkout -q {HARNESS_COMMIT}")
print("HarnessLLM @", subprocess.run(f"cd {HARNESS_DIR} && git rev-parse HEAD", shell=True,
                                     capture_output=True, text=True).stdout.strip())
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# === Cell 2: Cấu hình (đổi VARIANT mỗi phiên) ===
import os, json, re, random, glob, time, copy, math
def E(k, d): return type(d)(os.environ.get(k, d))

VARIANT   = E("VARIANT", "concept_aux")   # base | harness_rl | sft | concept_aux | placebo_aux | concept_prompt
SEED      = E("SEED", 42)
BASE_MODEL = "Qwen/Qwen3-4B"                       # = model_name_or_path trong scripts/qwen3_sft.yaml của paper
RL_MODEL   = "Shiyu-Lab/HarnessLLM_RL_Qwen3_4B"
SFT_DATA   = "Shiyu-Lab/HarnessLLM_SFT_Data"
EVAL_DATA  = E("EVAL_DATA", "Shiyu-Lab/Testcase_LCB_Seen")  # hoặc _LCB_Unseen, _CF_Seen, _CF_Unseen, _MBPPHard

# --- train (QLoRA) ---
N_TRAIN    = E("N_TRAIN", 800)       # ~1.5-2h / epoch trên 1×T4 với MAX_LEN 3072
N_PROBE    = E("N_PROBE", 60)        # held-out để đo concept head (chỉ biến thể *_aux)
MAX_LEN    = E("MAX_LEN", 3072)      # ví dụ dài hơn bị BỎ (không cắt, tránh dạy harness cụt)
EPOCHS     = E("EPOCHS", 1)
LR         = E("LR", 1e-4)
GRAD_ACC   = E("GRAD_ACC", 8)
LORA_R     = E("LORA_R", 16)
CONCEPT_WEIGHT = E("CONCEPT_WEIGHT", 0.2)   # ConceptCoder: concept_weight=0.20
MAX_STATEMENTS = 150
THINK      = E("THINK", 0)           # 0: tắt thinking của Qwen3 (bỏ <think>..</think> khỏi target SFT)

# --- eval ---
N_EVAL     = E("N_EVAL", 100)        # 0 = toàn bộ eval set
EVAL_SUBSET_SEED = 1234              # CỐ ĐỊNH để mọi VARIANT/SEED chấm cùng một tập bài
N_SAMPLES  = E("N_SAMPLES", 1)
MAX_NEW_TOKENS = E("MAX_NEW_TOKENS", 2048)
GEN_BATCH  = E("GEN_BATCH", 8)
TEMPERATURE = 0.6                    # như scripts/generate.py của paper (top_p 0.95)
N_PROC     = E("N_PROC", max(1, (os.cpu_count() or 2)))
INFER_4BIT = E("INFER_4BIT", 0)      # 1 nếu sinh fp16 bị NaN/inf trên T4: sinh bằng base 4-bit + adapter (chậm hơn)

TRAINABLE = {"sft", "concept_aux", "placebo_aux", "concept_prompt"}
assert VARIANT in TRAINABLE | {"base", "harness_rl"}, VARIANT
TAG = f"{VARIANT}_s{SEED}"
ADAPTER_DIR = os.path.join(WORK, f"adapter_{TAG}")

import torch
CAP = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
CDTYPE = torch.bfloat16 if CAP[0] >= 8 else torch.float16   # T4/P100: fp16
random.seed(SEED); torch.manual_seed(SEED)
print(f"VARIANT={VARIANT} SEED={SEED} TAG={TAG} dtype={CDTYPE} eval={EVAL_DATA} N_EVAL={N_EVAL}")

In [ ]:
# === Cell 3: Concept annotator (AST, 8 nhãn bug-prone / dòng, tương tự 8 nhãn tree-sitter của ConceptCoder VD) ===
import ast
CONCEPTS = ["boundary_compare",  # <, <=, >, >=  (off-by-one)
            "range_bound",       # range(..)/slice có số học hoặc len()
            "index_arith",       # a[i+1], a[-1], a[n-k]
            "div_mod",           # / // % divmod pow(.,.,m)
            "special_case",      # if x==0/1/-1/2, not x, len(x)==0 ... rồi return/break/print
            "extreme_init",      # 10**9, float('inf'), sys.maxsize, [inf]*n
            "order_select",      # sorted/.sort/min/max/heapq/bisect
            "io_parse_format"]   # input/stdin/split/int/map/print/join/format
K = len(CONCEPTS)
_SMALL = {0, 1, -1, 2}

def _dotted(e):
    if isinstance(e, ast.Name): return e.id
    if isinstance(e, ast.Attribute):
        b = _dotted(e.value); return (b + "." if b else ".") + e.attr
    return ""
def _cname(c): return _dotted(c.func)
def _arith(e): return any(isinstance(x, (ast.BinOp, ast.UnaryOp)) for x in ast.walk(e))
def _haslen(e): return any(isinstance(x, ast.Call) and _cname(x) == "len" for x in ast.walk(e))
def _num(e):
    if isinstance(e, ast.Constant) and isinstance(e.value, (int, float)) and not isinstance(e.value, bool): return e.value
    if isinstance(e, ast.UnaryOp) and isinstance(e.op, ast.USub):
        v = _num(e.operand); return -v if v is not None else None
    if isinstance(e, ast.BinOp) and isinstance(e.op, ast.Pow):
        a, b = _num(e.left), _num(e.right)
        if a is not None and b is not None and abs(b) < 64:
            try: return a ** b
            except Exception: return None
    return None
def _extreme(e):
    v = _num(e)
    if v is not None: return abs(v) >= 10 ** 9
    if isinstance(e, ast.Call) and _cname(e) == "float" and e.args and isinstance(e.args[0], ast.Constant):
        return str(e.args[0].value).lower().lstrip("+-") in ("inf", "infinity")
    if isinstance(e, (ast.Name, ast.Attribute)): return _dotted(e) in ("sys.maxsize", "math.inf")
    if isinstance(e, ast.BinOp) and isinstance(e.op, ast.Mult):
        return any(isinstance(s, ast.List) and any(_extreme(x) for x in s.elts) for s in (e.left, e.right))
    return False
def _special_test(t):
    if isinstance(t, ast.UnaryOp) and isinstance(t.op, ast.Not): return True
    if isinstance(t, ast.BoolOp): return any(_special_test(v) for v in t.values)
    if isinstance(t, ast.Compare):
        return any(_num(v) in _SMALL for v in [t.left] + list(t.comparators) if _num(v) is not None)
    return False
def _exits(body):
    for st in body:
        for x in ast.walk(st):
            if isinstance(x, (ast.Return, ast.Break, ast.Continue)): return True
            if isinstance(x, ast.Call) and _cname(x) in ("print", "exit", "sys.exit", "quit"): return True
    return False
_IO = {"input", "print", "int", "map", "open"}
_ORDER = {"sorted", "min", "max"}

def annotate_program(code):
    """-> list[len(lines)] of K-dim 0/1 vectors."""
    n = len(code.split("\n")); lab = [[0] * K for _ in range(n)]
    try: tree = ast.parse(code)
    except Exception: return lab
    def mark(node, k):
        s = getattr(node, "lineno", None)
        if s is None: return
        for i in range(s - 1, (getattr(node, "end_lineno", None) or s)):
            if 0 <= i < n: lab[i][k] = 1
    for x in ast.walk(tree):
        if isinstance(x, ast.Compare) and any(isinstance(o, (ast.Lt, ast.LtE, ast.Gt, ast.GtE)) for o in x.ops): mark(x, 0)
        if isinstance(x, ast.Call):
            nm = _cname(x); last = nm.split(".")[-1]
            if nm == "range" and any(_arith(a) or _haslen(a) for a in x.args): mark(x, 1)
            if nm == "divmod" or (nm == "pow" and len(x.args) == 3): mark(x, 3)
            if nm in _ORDER or last == "sort" or nm.startswith(("heapq.", "bisect.")): mark(x, 6)
            if nm in _IO or nm.startswith("sys.stdin") or last in ("split", "join", "format", "readline", "strip", "write"): mark(x, 7)
        if isinstance(x, ast.Slice) and any(v is not None and (_arith(v) or _haslen(v)) for v in (x.lower, x.upper)): mark(x, 1)
        if isinstance(x, ast.Subscript) and not isinstance(x.slice, ast.Slice):
            sl = x.slice
            if _arith(sl) or (_num(sl) is not None and _num(sl) < 0): mark(x, 2)
        if isinstance(x, ast.BinOp) and isinstance(x.op, (ast.Div, ast.FloorDiv, ast.Mod)): mark(x, 3)
        if isinstance(x, ast.AugAssign) and isinstance(x.op, (ast.Div, ast.FloorDiv, ast.Mod)): mark(x, 3)
        if isinstance(x, ast.If) and _special_test(x.test) and _exits(x.body): mark(x.test, 4)
        if isinstance(x, (ast.Assign, ast.AnnAssign)) and x.value is not None and _extreme(x.value): mark(x, 5)
    return lab

def render_concepts(code, lab, max_lines=12):
    lines = code.split("\n"); rows = []
    for i, (ln, v) in enumerate(zip(lines, lab)):
        tags = [CONCEPTS[k] for k in range(K) if v[k]]
        if tags and ln.strip(): rows.append((-sum(v[:7]), i, tags, ln.strip()[:120]))
    rows = sorted(sorted(rows)[:max_lines], key=lambda r: r[1])
    if not rows: return ""
    out = ["Static concept hints (bug-prone lines of the program):"]
    out += [f"- L{i + 1} [{', '.join(t)}]: {s}" for _, i, t, s in rows]
    out.append("Target these spots (boundaries, special cases, extremes, ordering, I/O format) in generate_input_* and check_output.")
    return "\n".join(out)

_demo = "n=int(input())\na=list(map(int,input().split()))\nbest=float('inf')\nfor i in range(n-1):\n    if a[i+1]-a[i] < best:\n        best=a[i+1]-a[i]\nif n==1:\n    print(0)\nelse:\n    print(best%1000000007)"
for ln, v in zip(_demo.split("\n"), annotate_program(_demo)):
    print(f"{ln:<38}", [CONCEPTS[k] for k in range(K) if v[k]])
print(render_concepts(_demo, annotate_program(_demo)))

In [ ]:
# === Cell 4: Nạp data SFT của HarnessLLM -> (prompt, target, program, nhãn concept) ===
from datasets import load_dataset
DO_TRAIN = VARIANT in TRAINABLE
_prev = glob.glob(f"/kaggle/input/**/adapter_{TAG}/adapter_config.json", recursive=True)
if DO_TRAIN and _prev:
    ADAPTER_DIR = os.path.dirname(_prev[0]); DO_TRAIN = False
    print("[resume] dùng adapter có sẵn:", ADAPTER_DIR)

_ROLE = {"human": "user", "gpt": "assistant", "model": "assistant", "bot": "assistant"}
def to_triplet(row):
    """Chấp nhận format sharegpt/messages/alpaca/prompt-response -> (system, user, assistant)."""
    for key in ("messages", "conversations", "conversation", "chat"):
        if row.get(key):
            msgs = []
            for m in row[key]:
                role = m.get("role", m.get("from")); content = m.get("content", m.get("value"))
                msgs.append((_ROLE.get(role, role), content))
            sys_ = next((c for r, c in msgs if r == "system"), row.get("system"))
            us = [c for r, c in msgs if r == "user"]; asst = [c for r, c in msgs if r == "assistant"]
            return (sys_, us[-1], asst[-1]) if us and asst else None
    if "instruction" in row and "output" in row:
        u = row["instruction"] + (("\n" + row["input"]) if row.get("input") else "")
        return row.get("system"), u, row["output"]
    for pk, rk in (("prompt", "response"), ("prompt", "completion"), ("prompt", "chosen"), ("question", "answer"), ("query", "response")):
        if pk in row and rk in row:
            p, r = row[pk], row[rk]
            if isinstance(p, list): p = p[-1]["content"]
            if isinstance(r, list): r = r[-1]["content"]
            return row.get("system"), p, r
    return None

def extract_program(user):
    blocks = re.findall(r"```python\n(.*?)```", user, flags=re.S)
    return blocks[-1].rstrip("\n") if blocks else None   # block cuối = chương trình cần test (block đầu = khung harness)

def clean_target(a):
    if not THINK: a = re.sub(r"<think>.*?</think>\s*", "", a, flags=re.S)
    return a.strip()

TRAIN_EX, PROBE_EX = [], []
if DO_TRAIN:
    raw = load_dataset(SFT_DATA, split="train")
    print(raw); print("columns:", raw.column_names)
    ex = []
    for i, row in enumerate(raw):
        t = to_triplet(row)
        if t is None:
            if i == 0: raise ValueError(f"Không nhận ra schema: {raw.column_names}. Sửa to_triplet().")
            continue
        sys_, user, asst = t; prog = extract_program(user); tgt = clean_target(asst)
        if not prog or "```python" not in tgt or "def generate_input_" not in tgt: continue
        lab = annotate_program(prog)
        ex.append({"uid": i, "system": sys_, "user": user, "target": tgt, "program": prog,
                   "true_labels": lab, "labels_concept": [r[:] for r in lab]})
    random.Random(SEED).shuffle(ex)
    print(f"usable {len(ex)}/{len(raw)}")
    dens = [sum(r[k] for e in ex for r in e["true_labels"]) / max(1, sum(len(e["true_labels"]) for e in ex)) for k in range(K)]
    print("concept density/line:", {c: round(d, 3) for c, d in zip(CONCEPTS, dens)})
    print("--- ví dụ target ---\n", ex[0]["target"][:800])
    POOL = ex

In [ ]:
# === Cell 5: Tokenize (chat template Qwen3) + map token -> dòng của chương trình ===
from bisect import bisect_right
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(RL_MODEL if VARIANT == "harness_rl" else BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
END = tok.eos_token if tok.eos_token else "<|im_end|>"

def chat_prompt(system, user):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    try: return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=bool(THINK))
    except TypeError: return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def user_text(e):
    if VARIANT == "concept_prompt":
        h = render_concepts(e["program"], e["true_labels"])
        return e["user"] + ("\n\n" + h if h else "")
    return e["user"]

def encode(e):
    p = chat_prompt(e["system"], user_text(e)); full = p + e["target"] + END
    enc = tok(full, add_special_tokens=False, return_offsets_mapping=True)
    ids, offs = enc["input_ids"], enc["offset_mapping"]
    if len(ids) > MAX_LEN: return None
    labels = [-100 if s < len(p) else t for t, (s, _) in zip(ids, offs)]   # chỉ học phần harness
    j = p.rfind("```python\n" + e["program"])
    line_map = [-1] * len(ids)
    if j >= 0:
        st = j + len("```python\n"); en = st + len(e["program"])
        starts = [st] + [st + k + 1 for k, ch in enumerate(e["program"]) if ch == "\n"]
        for t, (s, _) in enumerate(offs):
            if st <= s < en:
                ln = bisect_right(starts, s) - 1
                if ln < MAX_STATEMENTS: line_map[t] = ln
    return {"input_ids": ids, "labels": labels, "line_map": line_map,
            "concept_labels": e["labels_concept"], "true_labels": e["true_labels"]}

if DO_TRAIN:
    enc_rows = []
    for e in POOL:
        r = encode(e)
        if r is not None: enc_rows.append(r)
        if len(enc_rows) >= N_TRAIN + N_PROBE: break
    PROBE_ROWS, TRAIN_ROWS = enc_rows[:N_PROBE], enc_rows[N_PROBE:]
    if VARIANT == "placebo_aux":   # xáo nhãn concept giữa mọi dòng của tập train: giữ phân phối, phá căn chỉnh dòng↔nhãn
        flat = [row for r in TRAIN_ROWS for row in r["concept_labels"]]
        random.Random(SEED + 1).shuffle(flat); it = iter(flat)
        for r in TRAIN_ROWS: r["concept_labels"] = [next(it) for _ in r["concept_labels"]]
    lens = sorted(len(r["input_ids"]) for r in TRAIN_ROWS)
    print(f"train {len(TRAIN_ROWS)} | probe {len(PROBE_ROWS)} | tokens median {lens[len(lens)//2]} max {lens[-1]}")
    print("có line_map:", sum(any(x >= 0 for x in r["line_map"]) for r in TRAIN_ROWS), "/", len(TRAIN_ROWS))

In [ ]:
# === Cell 6: Train QLoRA (+ concept head kiểu ConceptCoder cho *_aux) ===
import gc
from torch import nn
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
USE_AUX = VARIANT in ("concept_aux", "placebo_aux")

def pool_lines(h, line_map):
    """ConceptCoder average_pool_by_segments: mean hidden state của các token thuộc mỗi dòng."""
    lm = torch.tensor(line_map, device=h.device); m = lm >= 0
    if int(m.sum()) == 0: return None, None
    idx = lm[m]; hs = h[m].float(); L = int(idx.max()) + 1
    cnt = torch.bincount(idx, minlength=L)
    pooled = torch.zeros(L, hs.size(-1), device=h.device).index_add_(0, idx, hs) / cnt.clamp(min=1).unsqueeze(1)
    return pooled, cnt > 0

def concept_loss(h, r, head, key="concept_labels"):
    pooled, valid = pool_lines(h, r["line_map"])
    if pooled is None: return None, None
    y = torch.tensor(r[key][:pooled.size(0)], device=h.device, dtype=torch.float32)
    logits = head(pooled)
    return nn.functional.binary_cross_entropy_with_logits(logits[valid], y[valid]), (logits[valid], y[valid])

def train():
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=CDTYPE)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0}, torch_dtype=CDTYPE)
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()
    head = nn.Linear(model.config.hidden_size, K).to("cuda")          # = `cbl` của ConceptCoder
    params = [p for p in model.parameters() if p.requires_grad]
    groups = [{"params": params, "lr": LR}] + ([{"params": list(head.parameters()), "lr": LR * 10}] if USE_AUX else [])
    opt = torch.optim.AdamW(groups, weight_decay=0.0)
    total = math.ceil(len(TRAIN_ROWS) * EPOCHS / GRAD_ACC)
    sch = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * total)), total)
    scaler = torch.amp.GradScaler("cuda", enabled=(CDTYPE == torch.float16))
    model.train(); hist = []; micro = 0; t0 = time.time(); acc_lm = acc_cl = 0.0; n_acc = 0
    for ep in range(EPOCHS):
        order = list(range(len(TRAIN_ROWS))); random.Random(SEED + ep).shuffle(order)
        for j, i in enumerate(order):
            r = TRAIN_ROWS[i]
            ids = torch.tensor([r["input_ids"]], device="cuda"); lab = torch.tensor([r["labels"]], device="cuda")
            with torch.autocast("cuda", dtype=CDTYPE):
                out = model(input_ids=ids, labels=lab, output_hidden_states=USE_AUX)
            loss = out.loss.float(); acc_lm += loss.item()
            if USE_AUX:
                cl, _ = concept_loss(out.hidden_states[-1][0], r, head)
                if cl is not None: loss = loss + CONCEPT_WEIGHT * cl; acc_cl += cl.item()
            scaler.scale(loss / GRAD_ACC).backward(); micro += 1; n_acc += 1
            if micro % GRAD_ACC == 0 or j == len(order) - 1:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params + list(head.parameters()), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step()
                step = micro // GRAD_ACC
                hist.append({"step": step, "lm": acc_lm / n_acc, "concept": acc_cl / n_acc})
                if step % 5 == 0 or j == len(order) - 1:
                    el = time.time() - t0
                    print(f"ep{ep} step {step}/{total} lm {acc_lm/n_acc:.4f} concept {acc_cl/n_acc:.4f} | {el/60:.1f}m, ETA {el/(j+1+ep*len(order))*(len(order)*EPOCHS-(j+1+ep*len(order)))/60:.0f}m")
                acc_lm = acc_cl = 0.0; n_acc = 0
            del out, loss
    # concept probe trên held-out, so với nhãn THẬT (placebo phải thấp)
    probe = None
    if USE_AUX and PROBE_ROWS:
        model.eval(); P = []; Y = []
        with torch.no_grad():
            for r in PROBE_ROWS:
                with torch.autocast("cuda", dtype=CDTYPE):
                    out = model(input_ids=torch.tensor([r["input_ids"]], device="cuda"), output_hidden_states=True)
                _, ly = concept_loss(out.hidden_states[-1][0], r, head, key="true_labels")
                if ly is not None: P.append((ly[0] > 0).float().cpu()); Y.append(ly[1].cpu())
        P = torch.cat(P); Y = torch.cat(Y)
        def f1(p, y):
            tp = (p * y).sum().item(); fp = (p * (1 - y)).sum().item(); fn = ((1 - p) * y).sum().item()
            return 2 * tp / max(1e-9, 2 * tp + fp + fn)
        probe = {"micro_f1": f1(P, Y), **{c: f1(P[:, k], Y[:, k]) for k, c in enumerate(CONCEPTS)}}
        print("concept probe (held-out, true labels):", {k: round(v, 3) for k, v in probe.items()})
    model.save_pretrained(ADAPTER_DIR); torch.save(head.state_dict(), os.path.join(ADAPTER_DIR, "concept_head.pt"))
    json.dump({"hist": hist, "probe": probe, "n_train": len(TRAIN_ROWS), "minutes": (time.time() - t0) / 60},
              open(os.path.join(ADAPTER_DIR, "train_log.json"), "w"))
    del model, head, opt; gc.collect(); torch.cuda.empty_cache()
    return probe

PROBE = train() if DO_TRAIN else None
if not DO_TRAIN and os.path.exists(os.path.join(ADAPTER_DIR, "train_log.json")):
    PROBE = json.load(open(os.path.join(ADAPTER_DIR, "train_log.json"))).get("probe")

In [ ]:
# === Cell 7: Sinh harness trên eval set (prompt `prompts/harness.py` của paper) ===
import importlib.util
from datasets import load_dataset
_spec = importlib.util.spec_from_file_location("harness_prompt", os.path.join(HARNESS_DIR, "prompts", "harness.py"))
_hp = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_hp)
MESSAGE_TEMPLATE = _hp.MESSAGE_TEMPLATE

ev = load_dataset(EVAL_DATA, split="train")
idx = list(range(len(ev)))
if N_EVAL and N_EVAL < len(ev):
    random.Random(EVAL_SUBSET_SEED).shuffle(idx); idx = sorted(idx[:N_EVAL])
EVAL_ITEMS = []
for i in idx:
    it = ev[i]; rm = it["reward_model"]; rm = json.loads(rm) if isinstance(rm, str) else rm
    gt_s = rm["ground_truth"]; gt = json.loads(gt_s) if isinstance(gt_s, str) else gt_s
    bad = gt["bad_code"]["code"] if isinstance(gt["bad_code"], dict) else gt["bad_code"]
    msgs = copy.deepcopy(MESSAGE_TEMPLATE[gt.get("code_type", "input/output")])
    msgs[-1]["content"] = msgs[-1]["content"].format(description=it["description"].strip(), testing_code=bad.strip())
    user = msgs[-1]["content"]
    if VARIANT == "concept_prompt":
        h = render_concepts(bad.strip(), annotate_program(bad.strip())); user += ("\n\n" + h if h else "")
    EVAL_ITEMS.append({"uid": str(it.get("uid", i)), "prompt": chat_prompt(None, user),
                       "ground_truth": gt_s if isinstance(gt_s, str) else json.dumps(gt_s)})
print(f"eval items: {len(EVAL_ITEMS)} / {len(ev)}  ({EVAL_DATA})")

GEN_PATH = os.path.join(WORK, f"gens_{TAG}.jsonl")
def generate_all():
    name = RL_MODEL if VARIANT == "harness_rl" else BASE_MODEL
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=CDTYPE) if INFER_4BIT else None
    m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=CDTYPE, device_map={"": 0}, quantization_config=q)
    if VARIANT in TRAINABLE:
        m = PeftModel.from_pretrained(m, ADAPTER_DIR)
        if not INFER_4BIT: m = m.merge_and_unload()   # merge LoRA vào base fp16 để sinh nhanh
    m.eval(); tok.padding_side = "left"
    order = sorted(range(len(EVAL_ITEMS)), key=lambda i: len(EVAL_ITEMS[i]["prompt"]))
    outs = {}; t0 = time.time()
    for b in range(0, len(order), GEN_BATCH):
        bi = order[b:b + GEN_BATCH]
        enc = tok([EVAL_ITEMS[i]["prompt"] for i in bi], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        torch.manual_seed(SEED + b)
        with torch.no_grad():
            g = m.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=0.95,
                           num_return_sequences=N_SAMPLES, pad_token_id=tok.pad_token_id)
        new = g[:, enc["input_ids"].shape[1]:]
        for k, i in enumerate(bi):
            res = []
            for s in range(N_SAMPLES):
                row = new[k * N_SAMPLES + s]
                txt = tok.decode(row, skip_special_tokens=True)
                txt = re.sub(r"<think>.*?</think>\s*", "", txt, flags=re.S).strip()
                res.append({"text": txt, "truncated": bool(tok.eos_token_id not in row.tolist())})
            outs[i] = res
        print(f"gen {min(b + GEN_BATCH, len(order))}/{len(order)} | {(time.time() - t0)/60:.1f}m")
    with open(GEN_PATH, "w") as f:
        for i, it in enumerate(EVAL_ITEMS):
            f.write(json.dumps({"uid": it["uid"], "samples": outs[i]}) + "\n")
    del m; gc.collect(); torch.cuda.empty_cache()

if os.path.exists(GEN_PATH): print("[resume] đã có", GEN_PATH)
else: generate_all()
GENS = {d["uid"]: d["samples"] for d in map(json.loads, open(GEN_PATH))}
print(GENS[EVAL_ITEMS[0]["uid"]][0]["text"][:1500])

In [ ]:
# === Cell 8: Chấm bằng judge GỐC của HarnessLLM (compute_score), mỗi harness chạy trong 1 process riêng ===
import multiprocessing as mp, types
sys.path.insert(0, HARNESS_DIR)
try:
    import prctl  # noqa
except ImportError:   # không build được python-prctl: shim no-op (chỉ mất tính năng kill process mồ côi)
    print("[warn] python-prctl không có -> dùng shim no-op cho prctl.set_pdeathsig")
    sys.modules["prctl"] = types.SimpleNamespace(set_pdeathsig=lambda *a, **k: None)
import custom_verl.testcase_judge as TJ
import tempfile
# reliability_guard() (gọi trong compute_score) đặt os.getcwd=None -> tempfile hỏng nếu tempdir chưa được cache.
# Worker ray của paper đã cache sẵn; ở đây cache trong process cha trước khi fork (không sửa code của họ).
tempfile.gettempdir()

def _worker(conn, text, gt):
    # tham số giống get_score_remote() trong scripts/eval.py của paper
    try:
        TJ.GLOBAL_TIMEOUT = 20
        score, rt, fr, ni = TJ.compute_score(text, gt, num_testcases=20, test_harness=True, num_repeats=1,
                                             timeout=20, seed=42, num_generators=5)
        f0 = fr[0]
        reason = f0[1].name if isinstance(f0, tuple) and hasattr(f0[1], "name") else str(f0)
        n_in = ni[0] + ni[1] if isinstance(ni, tuple) else ni
        conn.send((float(score), rt[0][0].name, reason, n_in))
    except Exception as e:
        conn.send((None, "Crash", repr(e)[:300], -1))
    finally:
        conn.close()

def score_all(tasks, nproc=N_PROC, hard_timeout=600):
    ctx = mp.get_context("fork"); res = [None] * len(tasks); running = {}; nxt = 0; t0 = time.time(); done = 0
    while nxt < len(tasks) or running:
        while nxt < len(tasks) and len(running) < nproc:
            a, b = ctx.Pipe(duplex=False); p = ctx.Process(target=_worker, args=(b,) + tasks[nxt], daemon=True)
            p.start(); b.close(); running[nxt] = (p, a, time.time()); nxt += 1
        for i, (p, a, t) in list(running.items()):
            fin = None
            if a.poll():
                try: fin = a.recv()
                except EOFError: fin = (None, "Crash", "EOF", -1)
            elif not p.is_alive(): fin = (None, "Crash", f"exitcode {p.exitcode}", -1)
            elif time.time() - t > hard_timeout:
                fin = (-0.1, "FailGood", "TimeoutError", -1)   # như nhánh GetTimeoutError trong eval.py
            if fin is not None:
                if p.is_alive(): p.kill()
                p.join(1); res[i] = fin; del running[i]; done += 1
                if done % 20 == 0: print(f"scored {done}/{len(tasks)} | {(time.time() - t0)/60:.1f}m")
        time.sleep(0.05)
    return res

tasks, keys = [], []
for it in EVAL_ITEMS:
    for s, smp in enumerate(GENS[it["uid"]]):
        tasks.append((smp["text"], it["ground_truth"])); keys.append((it["uid"], s, smp["truncated"]))
scores = score_all(tasks)

from collections import Counter, defaultdict
per = defaultdict(list)
for (uid, s, trunc), (sc, rt, reason, n_in) in zip(keys, scores):
    per[uid].append({"sample": s, "score": sc, "reward_type": rt, "fail_reason": reason, "num_inputs": n_in, "truncated": trunc})

def rates(per):
    """Trung bình theo bài (mỗi bài = trung bình qua N_SAMPLES)."""
    def frac(rows, keys_): return sum(r["reward_type"] in keys_ for r in rows) / len(rows)
    groups = {"bug_found(PassGoodFailBad)": {"PassGoodFailBad"},
              "wrong_test(FailGood)": {"FailGood", "GoodInputFailGood"},
              "good_input": {"GoodInputFailGood", "GoodInputPassGoodPassBad", "PassGoodFailBad"},
              "PassGoodPassBad": {"PassGoodPassBad", "GoodInputPassGoodPassBad"},
              "format_error": {"FormatError"}, "crash": {"Crash"}}
    return {g: 100 * sum(frac(rows, ks) for rows in per.values()) / len(per) for g, ks in groups.items()}

SUMMARY = rates(per)
print(f"\n=== {TAG} on {EVAL_DATA} ({len(per)} bài × {N_SAMPLES} mẫu) ===")
for k, v in SUMMARY.items(): print(f"{k:28s} {v:6.1f}%")
print("truncated:", f"{100*sum(k[2] for k in keys)/len(keys):.1f}%")
print("reward types:", Counter(r["reward_type"] for rows in per.values() for r in rows))

RES_PATH = os.path.join(WORK, f"results_{TAG}.json")
json.dump({"tag": TAG, "variant": VARIANT, "seed": SEED, "eval_data": EVAL_DATA, "n_eval": len(per),
           "eval_subset_seed": EVAL_SUBSET_SEED, "harness_commit": HARNESS_COMMIT, "summary": SUMMARY, "probe": PROBE,
           "config": {k: globals()[k] for k in ["BASE_MODEL", "N_TRAIN", "MAX_LEN", "EPOCHS", "LR", "GRAD_ACC", "LORA_R",
                                               "CONCEPT_WEIGHT", "THINK", "N_SAMPLES", "MAX_NEW_TOKENS", "TEMPERATURE"]},
           "per_item": per}, open(RES_PATH, "w"))
print("saved", RES_PATH)

In [ ]:
# === Cell 9: Ví dụ một harness bắt được bug / không bắt được ===
for want in ("PassGoodFailBad", "PassGoodPassBad", "FailGood"):
    hit = next(((uid, r) for uid, rows in per.items() for r in rows if r["reward_type"] == want), None)
    if hit:
        uid, r = hit; print("=" * 70, f"\n{want}  uid={uid}  fail_reason={r['fail_reason']}\n", "=" * 70)
        print(GENS[uid][r["sample"]]["text"][:1800])

In [ ]:
# === Cell 10: GỘP mọi results_*.json (phiên này + /kaggle/input từ phiên trước) và so sánh paired ===
import numpy as np
from scipy.stats import binomtest
files = sorted(set(glob.glob(os.path.join(WORK, "results_*.json")) + glob.glob("/kaggle/input/**/results_*.json", recursive=True)))
R = {}
for f in files:
    d = json.load(open(f))
    if d.get("eval_data") != EVAL_DATA: continue
    R.setdefault(d["tag"], d)
print(f"{len(R)} kết quả trên {EVAL_DATA}:", sorted(R))

def item_scores(d):   # tỉ lệ bắt bug theo từng bài
    return {u: np.mean([r["reward_type"] == "PassGoodFailBad" for r in rows]) for u, rows in d["per_item"].items()}

rows = []
for tag, d in sorted(R.items()):
    s = d["summary"]; pr = d.get("probe") or {}
    rows.append((tag, d["n_eval"], s["bug_found(PassGoodFailBad)"], s["wrong_test(FailGood)"], s["good_input"],
                 s["format_error"], pr.get("micro_f1")))
print(f"\n{'tag':24s} {'n':>4s} {'bug%':>6s} {'wrong%':>7s} {'goodin%':>8s} {'fmt%':>6s} {'probeF1':>8s}")
for t, n, b, w, g, fe, p in rows:
    print(f"{t:24s} {n:4d} {b:6.1f} {w:7.1f} {g:8.1f} {fe:6.1f} {('%.3f' % p) if p is not None else '-':>8s}")

def paired(a_tag, b_tag, B=5000, rng=np.random.default_rng(0)):
    A, Bm = item_scores(R[a_tag]), item_scores(R[b_tag]); common = sorted(set(A) & set(Bm))
    a = np.array([A[u] for u in common]); b = np.array([Bm[u] for u in common]); d = a - b
    boots = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(B)]
    n10 = int(((a > 0.5) & (b <= 0.5)).sum()); n01 = int(((a <= 0.5) & (b > 0.5)).sum())
    p = binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
    print(f"{a_tag:>20s} vs {b_tag:<20s} n={len(common):3d}  Δbug = {100*d.mean():+5.1f} pp "
          f"[95% CI {100*np.percentile(boots, 2.5):+5.1f}, {100*np.percentile(boots, 97.5):+5.1f}]  "
          f"win/lose {n10}/{n01}  McNemar p={p:.3f}")

print()
for a_, b_ in [("concept_aux", "sft"), ("concept_aux", "placebo_aux"), ("placebo_aux", "sft"), ("concept_prompt", "sft"),
               ("sft", "base"), ("concept_aux", "harness_rl"), ("harness_rl", "base")]:
    for s in sorted({d["seed"] for d in R.values()}):
        if f"{a_}_s{s}" in R and f"{b_}_s{s}" in R: paired(f"{a_}_s{s}", f"{b_}_s{s}")
        elif f"{a_}_s{s}" in R and f"{b_}_s42" in R and b_ in ("base", "harness_rl"): paired(f"{a_}_s{s}", f"{b_}_s42")